In [1]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, zscore
import hickle as hkl
from numpy.linalg import norm
from tqdm import tqdm

from sklearn.metrics import pairwise as kernel

from process_data import make_cell_embedding  # your existing 25Q3-aware function

MODEL_PATH = "datasets/2023/Model.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [3]:
def get_lung_model_ids(model_df: pd.DataFrame):
    """
    All lung cancers (SCLC + NSCLC + any Oncotree lung subtype),
    using OncotreePrimaryDisease / OncotreeSubtype and ModelID.
    """
    model_df[["OncotreeSubtype", "OncotreePrimaryDisease"]] = (
        model_df[["OncotreeSubtype", "OncotreePrimaryDisease"]].fillna("")
    )

    is_lung = (
        model_df["OncotreePrimaryDisease"].astype(str).str.lower().str.contains("lung")
        | model_df["OncotreeSubtype"].astype(str).str.lower().str.contains("lung")
    )

    return set(model_df.loc[is_lung, "ModelID"].astype(str))

In [4]:
# build global embedding + CRISPR via make_cell_embedding, then restrict to lung-only

# Use existing process_data pipeline (already tuned for 25Q3)
embedding_all, gene_effects_all = make_cell_embedding()
print("Global embedding:", embedding_all.shape)
print("Global CRISPR:",   gene_effects_all.shape)

# Load Model.csv and get lung IDs
model_df = pd.read_csv(MODEL_PATH)
lung_ids = pd.Index(get_lung_model_ids(model_df))

# Restrict embedding & CRISPR to lung-only cell lines
common_lung = (
    embedding_all.index
    .intersection(gene_effects_all.index)
    .intersection(lung_ids)
)

embedding = embedding_all.loc[common_lung].copy()
gene_effects_df = gene_effects_all.loc[common_lung].copy()

print("LUNG embedding:", embedding.shape)
print("LUNG CRISPR:",   gene_effects_df.shape)

# Save lung-only objects for later use (and for SCLC scripts)
hkl.dump(embedding,      "embeddings/final_X_tcga_lung_processed.hkl",         mode="w")
hkl.dump(gene_effects_df,"datasets/2023/CRISPRGeneEffect_lung_processed.hkl", mode="w")
print("Saved lung-only .hkl files")

# For the rest of the notebook, stick to these names (as in your original)
cell_embedding = embedding

Global embedding: (1112, 33587)
Global CRISPR: (1112, 18435)
LUNG embedding: (121, 33587)
LUNG CRISPR: (121, 18435)
pandas 2.3.3
Saved lung-only .hkl files


/opt/miniconda3/envs/augert/lib/python3.10/site-packages/hickle/lookup.py:1491: SerializedWarning: 'DataFrame' type not understood, data is serialized:
  warnings.warn(


In [5]:
# train individual RFM on lung-only data

def train_individual_rfm_cell():
    bandwidth = 1
    reg = 1e-5

    X = torch.tensor(cell_embedding.values).to(device).float()
    num_cells = X.shape[0]
    knockouts = gene_effects_df.columns
    num_knockouts = len(knockouts)

    # pairwise distances via sklearn (CPU) then convert to torch
    dists_np = kernel.euclidean_distances(cell_embedding.values, cell_embedding.values)
    cell_distances = torch.tensor(dists_np, device=device).float()
    dist_ko = cell_distances.fill_diagonal_(0)

    y = torch.tensor(gene_effects_df.values).to(device).float()

    sol = torch.linalg.solve(
        torch.exp(-bandwidth * (dist_ko)**0.5).to(device)
        + reg * torch.eye(dist_ko.shape[0], device=device),
        y
    )

    return sol

sol = train_individual_rfm_cell()
print("sol shape (n_cells x n_kos):", sol.shape)


sol shape (n_cells x n_kos): torch.Size([121, 18435])


In [6]:
# Euclidean distances + Laplace kernel (same math as your working notebook)

def euclidean_distances(samples, centers, M=None, squared=True, diag_only=False):
    """
    Torch version of pairwise squared Euclidean distances with optional diagonal metric M.
    """
    if M is None:
        samples_norm = torch.sum(samples**2, dim=1, keepdim=True)
    else:
        if diag_only:
            samples_norm = (samples * M) * samples
        else:
            samples_norm = (samples @ M) * samples
        samples_norm = torch.sum(samples_norm, dim=1, keepdims=True)

    if samples is centers:
        centers_norm = samples_norm
    else:
        if M is None:
            centers_norm = torch.sum(centers**2, dim=1, keepdims=True)
        else:
            if diag_only:
                centers_norm = (centers * M) * centers
            else:
                centers_norm = (centers @ M) * centers
            centers_norm = torch.sum(centers_norm, dim=1, keepdims=True)
    centers_norm = torch.reshape(centers_norm, (1, -1))

    distances = samples.mm(torch.t(centers))
    distances.mul_(-2)
    distances.add_(samples_norm)
    distances.add_(centers_norm)

    if not squared:
        distances.clamp_(min=0)
        distances.sqrt_()

    return distances


def laplace_kernel(samples, centers, bandwidth, M=None, diag_only=False):
    """
    Laplacian kernel K(x, x') = exp(-||x - x'|| / bandwidth)
    """
    assert bandwidth > 0
    kernel_mat = euclidean_distances(samples, centers, M=M, squared=False, diag_only=diag_only)
    kernel_mat.clamp_(min=0)
    gamma = 1.0 / bandwidth
    kernel_mat.mul_(-gamma)
    kernel_mat.exp_()
    return kernel_mat


In [7]:
# compute grads on lung-only data

def get_grads(X, sol, P, L=1, centering=False, diag_only=True):
    K = laplace_kernel(X, X, bandwidth=1, M=P, diag_only=diag_only)

    dist = euclidean_distances(X, X, M=P, squared=False, diag_only=diag_only)
    dist.clamp_(min=0)
    dist[dist < 1e-10] = 0

    with np.errstate(divide="ignore"):
        K = K / dist

    K[K == float("Inf")] = 0.0
    n, d = X.shape
    num_kos, n2 = sol.shape
    assert n == n2

    grads = torch.zeros((d, num_kos)).to(X.device)
    for i in tqdm(range(num_kos)):
        weight = sol[i, :].reshape((-1, 1))

        step2 = K @ (weight * X)
        step3 = (weight.T @ K).T * X
        G = (step2 - step3) * -1 / L
        G = torch.sum(G**2, axis=0)
        grads[:, i] = G / n

    return grads

X = torch.tensor(cell_embedding.values).to(device).float()
n, d = X.shape
P = torch.ones(d).double().to(device)

# sol is (n_cells x n_kos); get_grads expects (num_kos x n_cells)
grads_tensor = get_grads(X, sol.T, P=P, L=1, centering=False, diag_only=True)

print("grads_tensor shape (d x num_kos):", grads_tensor.shape)

100%|██████████| 18435/18435 [03:28<00:00, 88.43it/s]

grads_tensor shape (d x num_kos): torch.Size([33587, 18435])


In [8]:
# PCC computation (same fixed version you already had)

def get_pcc(cell_embedding_df, gene_effects_df):
    cell_embedding = cell_embedding_df.copy()
    cell_embedding /= norm(cell_embedding, axis=1).reshape(-1, 1)

    exp_cols = [e for e in cell_embedding.columns if e.split("_")[-1] == "exp"]

    std_val = cell_embedding[exp_cols].std(axis=0).replace(0, 1)
    zscore_vals = (cell_embedding[exp_cols] - cell_embedding[exp_cols].mean(axis=0)) / std_val
    cell_embedding[exp_cols] *= (np.abs(zscore_vals) < 3).fillna(0).astype(int)

    normalized_cell_embedding = cell_embedding - cell_embedding.mean(axis=0)
    normalized_gene_effects_df = gene_effects_df - gene_effects_df.mean(axis=0)

    cell_norms = (normalized_cell_embedding**2).sum(axis=0).values
    gene_norms = (normalized_gene_effects_df**2).sum(axis=0).values

    pcc = (normalized_cell_embedding.T @ normalized_gene_effects_df) / (
        cell_norms.reshape((-1, 1)) @ gene_norms.reshape((1, -1))
    ) ** 0.5

    features = normalized_cell_embedding.columns
    knockouts = normalized_gene_effects_df.columns

    pcc_df = pd.DataFrame(pcc, index=features, columns=knockouts)
    return pcc_df


In [9]:
# build grads DataFrame, compute PCC, and feature_importance_df (lung-only)

features  = cell_embedding.columns
knockouts = gene_effects_df.columns

grads = pd.DataFrame(
    grads_tensor.detach().cpu().numpy(),
    index=features,
    columns=knockouts,
)

print("grads DataFrame shape:", grads.shape)

pcc = get_pcc(cell_embedding, gene_effects_df).fillna(0)
print("pcc shape:", pcc.shape)

# Align pcc to grads rows
pcc = pcc.loc[grads.index]

mut_feats = [x for x in grads.index if not x.endswith("_exp")]
exp_feats = [x for x in grads.index if x.endswith("_exp")]

# Mutation features: flip sign of negative correlations (keep positive effect)
pcc.loc[mut_feats] = -(pcc.loc[mut_feats].clip(upper=0))
# Expression features: absolute value
pcc.loc[exp_feats] = pcc.loc[exp_feats].abs()

feature_importance_df = grads * pcc
print("feature_importance_df shape (lung):", feature_importance_df.shape)


grads DataFrame shape: (33587, 18435)
pcc shape: (33587, 18435)
feature_importance_df shape (lung): (33587, 18435)


In [10]:
# save lung-specific feature_importances as separate .npy files

np.save("datasets/feature_importances_lung_data.npy",   feature_importance_df.values)
np.save("datasets/feature_importances_lung_index.npy",  feature_importance_df.index.to_numpy())
np.save("datasets/feature_importances_lung_columns.npy",feature_importance_df.columns.to_numpy())

print("Saved lung-specific feature_importances_lung_*.npy files")


Saved lung-specific feature_importances_lung_*.npy files
